# Task 2: Weighted Data Cleaning and Preprocessing

This notebook prepares the 2025 SHED survey for analysis while preserving survey design. It retains respondents with structural skips, preserves survey weights, reports missingness and outliers, and excludes identifiers, weights, and imputation flags from model features.

Blank survey cells are represented as `__NOT_ASKED__` in the modeling copy because skip logic is different from ordinary missingness. The raw dataframe is never overwritten.

## 1. Imports and configuration

In [22]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder

DATA_PATH = Path('../data/public2025.csv')
EXPECTED_ROWS = 12_934
WEIGHT_COLUMNS = ['weight', 'weight_pop', 'panel_weight', 'panel_weight_pop']
ID_COLUMNS = ['shedid']
NOT_ASKED = '__NOT_ASKED__'

## 2. Load and validate the raw survey

In this part, load and validate the raw dataset. We are checking whether the required columns are present, respondent IDs are valid and unique, survey weights contain valid values, and the dataset has the expected number of rows.

### Load raw data and inspect dimensions

In [23]:
raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Raw shape: {raw.shape[0]:,} rows x {raw.shape[1]:,} columns')

Raw shape: 12,934 rows x 815 columns


### Validate required columns and respondent IDs

In [24]:
required = {'shedid', *WEIGHT_COLUMNS}

missing_required = required.difference(raw.columns)
assert not missing_required, f'Missing required columns: {sorted(missing_required)}'

assert raw['shedid'].notna().all(), 'Respondent IDs contain blanks'
assert raw['shedid'].is_unique, 'Respondent IDs are duplicated'

print("Required columns and respondent IDs passed validation.")

Required columns and respondent IDs passed validation.


### Validate survey weights
`weight_pop` is used for population estimates. The other survey weights remain available for alternate analyses.

In [25]:
assert raw[['weight', 'weight_pop']].notna().all().all(), \
    'Primary survey weights must be present'

weight_values = raw[WEIGHT_COLUMNS]

assert ((weight_values >= 0) | weight_values.isna()).all().all(), \
    'Present survey weights must be non-negative'

assert np.isfinite(
    weight_values.dropna().to_numpy(dtype=float)
).all(), 'Present survey weights must be finite'

print("Survey weights passed validation.")

Survey weights passed validation.


### Check dataset size
Compare the loaded dataset against the expected row count from the project brief.

In [26]:
if len(raw) != EXPECTED_ROWS:
    print(
        f'Note: expected {EXPECTED_ROWS:,} rows '
        f'from the project brief; found {len(raw):,}.'
    )
else:
    print('Row count matches the project brief.')

Row count matches the project brief.


## 3. Profile missingness and imputation flags

Before cleaning the data, we want to look at variable types and missingness across the dataset. We also want to keep track of the imputation flag columns (_iflag), which will be excluded from the model features later. We do not drop rows based on missing values at this stage.

In [27]:
iflag_columns = [column for column in raw.columns if column.endswith('_iflag')]

profile = pd.DataFrame({
    'dtype': raw.dtypes.astype(str),
    'blank_count': raw.isna().sum(),
    'blank_rate': raw.isna().mean(),
}).sort_values('blank_rate', ascending=False)

print(f'Imputation flag columns: {len(iflag_columns):,}')
display(profile.head(20))

Imputation flag columns: 357


,dtype,blank_count,blank_rate
BK49B,float64,12823,0.991418
A8_d,str,12818,0.991031
A8_c,str,12812,0.990567
A8_e,str,12808,0.990258
S18,str,12794,0.989176
R5C_c,str,12773,0.987552
R5C_b,str,12773,0.987552
R5C_d,str,12773,0.987552
R5C_a,str,12773,0.987552
S21,str,12655,0.978429


## 4. Preserve skip logic

Blank cells are not a reason to drop respondents. Existing response labels such as declined or don't know remain unchanged. The codebook should be consulted before recoding categories.

In [28]:
def mark_structural_skips(frame, columns=None, marker=NOT_ASKED):
    """Replace remaining blanks with a skip marker.

    Item nonresponse has already been imputed and flagged in the source
    data, so remaining blanks are treated as questions that were not asked.
    """
    result = frame.copy()
    if columns is None:
        protected = set(ID_COLUMNS + WEIGHT_COLUMNS)
        columns = [column for column in result.columns if column not in protected]
    for column in columns:
        if pd.api.types.is_numeric_dtype(result[column]):
            # Keep numeric columns numeric.
            # Remaining NaNs will be handled by the modeling imputer.
            continue
        else:
            # For categorical columns, explicitly label structural skips.
            result[column] = result[column].astype('object').where(
                result[column].notna(), marker
            )
    return result

In [29]:
clean = mark_structural_skips(raw)

assert len(clean) == len(raw), 'Cleaning unexpectedly removed respondents'
assert clean['shedid'].equals(raw['shedid']), 'Respondent IDs changed during cleaning'
assert clean[WEIGHT_COLUMNS].equals(raw[WEIGHT_COLUMNS]), 'Survey weights changed during cleaning'

print(f'Rows retained: {len(clean):,} of {len(raw):,}')
print(f'Blank cells remaining in modeling copy: {int(clean.isna().sum().sum()):,}')

Rows retained: 12,934 of 12,934
Blank cells remaining in modeling copy: 104,113


## 5. Select features without leakage

Imputation flags describe imputation and must not predict the corresponding survey answers. Protected demographic variables remain available for later subgroup auditing. State geography is excluded from this general-purpose feature matrix.

In [30]:
def feature_columns(frame, iflags):
    # create a list of columns that should not be given to the model
    excluded = set(ID_COLUMNS + WEIGHT_COLUMNS + iflags)

    # Remove survey metadata / administrative variables
    excluded.update([
        'duration',
        'xlaptop',
        'field_month',
        'field_day',
        'control',
        'year',
    ])
    
    # remove state-level geography
    excluded.update(column for column in frame.columns if column.lower() in {'ppstaten', 'state'})
    return [column for column in frame.columns if column not in excluded]

Selects the variables that can be safely used as model features. We are removing respondent IDs, survey weights, imputation flags, and state-level geography to prevent data leakage or inappropriate use of identifying/weighting information. The remaining variables are stored in X as the candidate feature set, with checks to make sure excluded variables were not accidentally included.

In [31]:
feature_names = feature_columns(clean, iflag_columns)
X = clean[feature_names].copy()

assert not set(X.columns).intersection(iflag_columns), \
    'Imputation flags leaked into features'

assert not set(X.columns).intersection(ID_COLUMNS + WEIGHT_COLUMNS), \
    'IDs or weights leaked into features'

EXCLUDED_METADATA = {
    'duration',
    'xlaptop',
    'field_month',
    'field_day',
    'control',
    'year',
}

assert not set(X.columns).intersection(EXCLUDED_METADATA), \
    'Excluded metadata leaked into features'

assert X.shape[1] == 446, \
    f'Expected 446 candidate features, found {X.shape[1]}'

print(f'Candidate features: {X.shape[1]:,}')

Candidate features: 446


## 6. Report numeric outliers

The IQR rule is a screening diagnostic, not proof that a survey response is invalid. No values are changed here; capping or transformation requires a codebook-based decision.

In [32]:
def iqr_outlier_report(frame):
    rows = []
    for column in frame.select_dtypes(include=np.number).columns:
        values = frame[column].dropna()
        if values.empty:
            continue
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        count = int(((values < lower) | (values > upper)).sum())
        rows.append({'column': column, 'q1': q1, 'q3': q3, 'lower_bound': lower, 'upper_bound': upper, 'outlier_count': count, 'outlier_rate': count / len(values)})
    return pd.DataFrame(rows).sort_values('outlier_count', ascending=False)

In [33]:
outlier_report = iqr_outlier_report(raw.drop(columns=WEIGHT_COLUMNS, errors='ignore'))
display(outlier_report.head(20))

,column,q1,q3,lower_bound,upper_bound,outlier_count,outlier_rate
14,ppcmdate,20250207.0,2.025031e+07,2.025005e+07,2.025046e+07,4382,0.338797
23,ppc2date,20250130.0,2.025032e+07,2.024985e+07,2.025060e+07,3411,0.320372
16,ppp2date,20250604.0,2.025073e+07,2.025042e+07,2.025091e+07,3060,0.260714
18,pph1date,20250412.0,2.025061e+07,2.025012e+07,2.025090e+07,2567,0.225988
1,duration,953.0,1.963000e+03,-5.620000e+02,3.478000e+03,1772,0.137003
0,shedid,202300108.5,2.025036e+08,2.019948e+08,2.028089e+08,1457,0.112649
10,pphhsize,2.0,3.000000e+00,5.000000e-01,4.500000e+00,1438,0.111180
26,pphhsize5,2.0,3.000000e+00,5.000000e-01,4.500000e+00,1438,0.111180
20,ppmpdate,20250428.0,2.025051e+07,2.025030e+07,2.025064e+07,1139,0.096878
130,GH14_iflag,0.0,0.000000e+00,0.000000e+00,0.000000e+00,971,0.075073


## 7. Weighted summaries

Here, we are checking the supplied population weights and calculates the effective sample size. It does not modify or remove observations. The effective sample size summarizes how much information the weighted sample contains relative to an equal-weight sample. Unequal weights generally reduce the effective sample size.

In [34]:
def effective_sample_size(weights):
    weights = pd.Series(weights, dtype='float64').dropna()
    return float(weights.sum() ** 2 / weights.pow(2).sum())

print(f"Sum of population weights: {raw['weight_pop'].sum():,.2f}")
print(f"Effective sample size: {effective_sample_size(raw['weight_pop']):,.1f}")

Sum of population weights: 264,519,125.02
Effective sample size: 11,307.0


Calculate population-weighted distributions for selected survey variables.
Both weighted and unweighted counts are retained for comparison.

In [35]:
def weighted_distribution(frame, column, weight_column='weight_pop'):
    values = frame[[column, weight_column]].dropna(subset=[weight_column]).copy()
    summary = values.groupby(column, dropna=False)[weight_column].agg(['sum', 'count']).rename(columns={'sum': 'weighted_count', 'count': 'unweighted_count'})
    summary['weighted_proportion'] = summary['weighted_count'] / summary['weighted_count'].sum()

    summary['weighted_count'] = summary['weighted_count'].round(0).astype(int)
    summary['weighted_proportion'] = summary['weighted_proportion'].round(4)

    return summary.sort_values('weighted_proportion', ascending=False)

for column in ['EF1', 'B2']:
    if column in raw.columns:
        print(f'Weighted distribution for {column}:')
        display(weighted_distribution(raw, column))

Weighted distribution for EF1:


,weighted_count,unweighted_count,weighted_proportion
EF1,,,
Yes,144864161,7327,0.5477
No,119654964,5607,0.4523


Weighted distribution for B2:


,weighted_count,unweighted_count,weighted_proportion
B2,,,
Doing okay,103096420,5005,0.3898
Living comfortably,89808619,4479,0.3395
Just getting by,49523665,2387,0.1872
Finding it difficult to get by,22090422,1063,0.0835


## 8. Build a preprocessing-ready matrix

Fit this transformer only on the training split when modeling to prevent preprocessing leakage.

### 8.1 Identify feature types

Separating the candidate features into numeric and categorical variables so that each type can receive an appropriate preprocessing step.

In [36]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

ORDINAL_FEATURES = [
    'B2',
    'inc_4cat_50k'
]

categorical_features = [
    column for column in categorical_features
    if column not in ORDINAL_FEATURES
]

assert set(ORDINAL_FEATURES).issubset(X.columns)
assert (
    len(numeric_features)
    + len(categorical_features)
    + len(ORDINAL_FEATURES)
    == X.shape[1]
)

print(f'Numeric features: {len(numeric_features):,}')
print(f'Categorical features: {len(categorical_features):,}')
print(f'Ordinal features: {len(ORDINAL_FEATURES):,}')

Numeric features: 23
Categorical features: 421
Ordinal features: 2


### 8.2 Define preprocessing pipelines

Numeric features use median imputation for any remaining missing values. Categorical features use the most frequent observed value for imputation and are then one-hot encoded. Unknown categories are ignored when transforming future data so that new category values do not cause an error.

In [37]:
try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=True)

numeric_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median'))])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('to_string', FunctionTransformer(lambda values: values.astype(str))),
    ('one_hot', encoder),
])

ordinal_categories = [
    [
        'Finding it difficult to get by',
        'Just getting by',
        'Doing okay',
        'Living comfortably',
    ],
    [
        'Less than $25,000',
        '$25,000–$49,999',
        '$50,000–$99,999',
        '$100,000 or more',
    ],
]
ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    )),
])

### Column Transformer

In [38]:
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
    ('ordinal', ordinal_pipeline, ORDINAL_FEATURES),
], remainder='drop')

### 8.3 Encode features

Fit the preprocessing pipeline to the candidate feature matrix and transform
the survey variables into a numerical sparse matrix suitable for machine
learning. The row-count check confirms that preprocessing did not remove
respondents.

In [39]:
X_encoded = preprocessor.fit_transform(X)

assert X_encoded.shape[0] == len(raw), 'Encoding removed respondents'
assert X_encoded.shape[1] > X.shape[1], \
    'Encoded feature count is unexpectedly small'

print(f'Encoded matrix shape: {X_encoded.shape[0]:,} rows x {X_encoded.shape[1]:,} columns')

Encoded matrix shape: 12,934 rows x 2,464 columns


Structural skips marked as NOT_ASKED are treated as a categorical response during encoding rather than being interpreted as missing data.

## 9. Cleaning report and final checks

Summarize the final dataset dimensions, feature types, survey-weight
diagnostics, and encoded feature count. Validation checks confirm that no
respondents were removed and that imputation-flag columns are excluded from
the modeling features.

In [40]:
cleaning_report = pd.Series({
    'raw_rows': len(raw),
    'raw_columns': raw.shape[1],
    'rows_retained': len(clean),
    'rows_dropped': len(raw) - len(clean),
    'iflag_columns': len(iflag_columns),
    'candidate_features': len(feature_names),
    'numeric_features': len(numeric_features),
    'categorical_features': len(categorical_features),
    'encoded_features': X_encoded.shape[1],
    'population_weight_sum': raw['weight_pop'].sum(),
    'effective_sample_size': effective_sample_size(raw['weight_pop']),
})
display(cleaning_report.to_frame('value'))

,value
raw_rows,1.293400e+04
raw_columns,8.150000e+02
rows_retained,1.293400e+04
rows_dropped,0.000000e+00
iflag_columns,3.570000e+02
candidate_features,4.460000e+02
numeric_features,2.300000e+01
categorical_features,4.210000e+02
encoded_features,2.464000e+03
population_weight_sum,2.645191e+08


In [41]:
assert cleaning_report['rows_dropped'] == 0
assert not set(feature_names).intersection(iflag_columns)

assert (
    len(numeric_features)
    + len(categorical_features)
    + len(ORDINAL_FEATURES)
    == len(feature_names)
)

print('Task 2 preprocessing checks passed.')

Task 2 preprocessing checks passed.


## 10. Export the cleaned respondent-level data

The original source file remains unchanged. This export contains the cleaned survey responses with structural skips marked as `__NOT_ASKED__`, while respondent IDs and survey weights are preserved.

In [45]:
EXCLUDED_METADATA = {
    'duration',
    'xlaptop',
    'field_month',
    'field_day',
    'control',
    'year',
}

clean_export = clean.drop(columns=EXCLUDED_METADATA)

clean_export.to_csv(CLEAN_OUTPUT_PATH, index=False)

assert CLEAN_OUTPUT_PATH.exists(), 'Clean CSV was not created'

exported = pd.read_csv(CLEAN_OUTPUT_PATH, low_memory=False)

assert exported.shape == clean_export.shape, \
    'Exported CSV shape does not match clean export dataframe'

assert exported['shedid'].equals(clean_export['shedid']), \
    'Exported respondent IDs do not match clean export dataframe'

assert exported['shedid'].nunique() == clean_export['shedid'].nunique(), \
    'Exported respondent ID count does not match clean export dataframe'

assert not set(exported.columns).intersection(EXCLUDED_METADATA), \
    'Excluded metadata columns are present in the exported CSV'

print(
    f'Wrote {exported.shape[0]:,} rows x {exported.shape[1]:,} columns '
    f'to {CLEAN_OUTPUT_PATH}'
)
print('Verified: all 6 excluded metadata columns are absent from the exported CSV')

Wrote 12,934 rows x 809 columns to ..\data\public2025_clean.csv
Verified: all 6 excluded metadata columns are absent from the exported CSV
